# Fine-Tuning Qwen2.5-VL-3B for 2D Floor Plan Detection (Colab)
End-to-end QLoRA pipeline: **dataset → fine-tune → evaluate on unseen test plans → 2D overlay + 3D extrusion**.

This notebook writes the project's `src/` files and runs them, so Colab and local runs use **identical**
prompt, data format, training code and metrics.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or better). Run the cells top to bottom.

## 1. Install dependencies & check GPU

In [ ]:
!pip install -q "transformers>=4.49.0" "accelerate>=0.34.0" "peft>=0.14.0" "bitsandbytes>=0.43.0" \
    "datasets>=2.19.0" "qwen-vl-utils>=0.0.8" pillow matplotlib

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Write project source files
(Same code as the repository's `src/` folder.)

In [ ]:
import os
os.makedirs("/content/floorplan/src", exist_ok=True)
os.makedirs("/content/floorplan/tests", exist_ok=True)
%cd /content/floorplan

In [ ]:
%%writefile src/schema.py
"""
Floor Plan Schema, Prompt and Validation Module.

Single source of truth for:
  * the instruction prompt used for training AND inference (they must match),
  * the JSON target format (compact JSON, no markdown fences),
  * schema validation of model output,
  * robust JSON extraction from raw model text,
  * project-relative path resolution (so scripts work from any working directory).
"""

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Project root = parent of the src/ folder
PROJECT_ROOT = Path(__file__).resolve().parent.parent

ROOM_TYPES = ["living_room", "bedroom", "kitchen", "bathroom", "hallway", "dining_room"]
WALL_TYPES = ["interior", "exterior"]
DOOR_TYPES = ["entrance", "interior", "sliding"]
CATEGORIES = ["rooms", "walls", "doors", "windows"]
EXPECTED_KEYS = set(CATEGORIES)

FLOORPLAN_PROMPT = """Analyze this 2D architectural floor plan image. Detect and classify all structural elements:
1. Rooms (living_room, bedroom, kitchen, bathroom, hallway, dining_room)
2. Walls (interior, exterior)
3. Doors (entrance, interior, sliding)
4. Windows

Return ONLY a valid JSON object (no markdown, no explanation). All coordinates are integers normalized to 0-1000.
Bounding boxes use [ymin, xmin, ymax, xmax]. Wall endpoints use [x, y].
Format:
{"image_dimensions": {"width": W, "height": H}, "rooms": [{"id": "room_1", "type": "bedroom", "bbox": [ymin, xmin, ymax, xmax]}], "walls": [{"id": "wall_1", "type": "exterior", "start": [x1, y1], "end": [x2, y2], "bbox": [ymin, xmin, ymax, xmax]}], "doors": [{"id": "door_1", "type": "interior", "bbox": [ymin, xmin, ymax, xmax]}], "windows": [{"id": "window_1", "bbox": [ymin, xmin, ymax, xmax]}]}"""


def to_target_text(ground_truth: Dict[str, Any]) -> str:
    """Serialize ground truth as the exact assistant target string (compact = fewer tokens)."""
    return json.dumps(ground_truth, separators=(",", ":"), ensure_ascii=False)


def resolve_path(path: str, base: Optional[Path] = None) -> str:
    """Resolve a (possibly relative) path: as given -> relative to base -> relative to project root."""
    if path.startswith(("file://", "http://", "https://", "data:")):
        return path
    p = Path(path)
    if p.is_absolute() and p.exists():
        return str(p)
    for root in [Path.cwd(), base, PROJECT_ROOT]:
        if root is not None and (Path(root) / p).exists():
            return str((Path(root) / p).resolve())
    return str(p)


def _is_valid_box(box: Any) -> bool:
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return False
    if not all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in box):
        return False
    ymin, xmin, ymax, xmax = box
    return ymin <= ymax and xmin <= xmax


def _is_valid_point(pt: Any) -> bool:
    return (
        isinstance(pt, (list, tuple)) and len(pt) == 2
        and all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in pt)
    )


def validate_floorplan_json(data: Any, coord_max: int = 1000) -> Tuple[bool, List[str]]:
    """
    Validates a parsed JSON object against the 2D-to-3D floor plan schema.
    Returns (is_valid, error_messages).
    """
    errors: List[str] = []
    if not isinstance(data, dict):
        return False, ["Root output is not a JSON object/dict."]

    missing = EXPECTED_KEYS - set(data.keys())
    if missing:
        errors.append(f"Missing top-level keys: {sorted(missing)}")

    def check_range(box, label):
        if any(v < 0 or v > coord_max for v in box):
            errors.append(f"{label} has coordinates outside [0, {coord_max}].")

    for cat in CATEGORIES:
        items = data.get(cat, [])
        if not isinstance(items, list):
            errors.append(f"'{cat}' must be a list.")
            continue
        for idx, item in enumerate(items):
            label = f"{cat}[{idx}]"
            if not isinstance(item, dict):
                errors.append(f"{label} is not an object.")
                continue
            label = f"{cat}[{item.get('id', idx)}]"

            if cat == "walls":
                has_bbox = _is_valid_box(item.get("bbox"))
                has_pts = _is_valid_point(item.get("start")) and _is_valid_point(item.get("end"))
                if not (has_bbox or has_pts):
                    errors.append(f"{label} needs a valid 'bbox' or 'start'/'end' endpoints.")
                if has_bbox:
                    check_range(item["bbox"], label)
            else:
                if not _is_valid_box(item.get("bbox")):
                    errors.append(f"{label} missing valid 4-number 'bbox' [ymin, xmin, ymax, xmax].")
                else:
                    check_range(item["bbox"], label)

            if cat == "rooms" and "type" not in item:
                errors.append(f"{label} missing 'type'.")

    return (len(errors) == 0, errors)


def extract_json_from_response(text: str) -> Dict[str, Any]:
    """
    Extracts and parses a JSON object from raw model output.
    Handles markdown fences, leading/trailing chatter and (best-effort) truncated output.
    Raises ValueError if nothing parseable is found.
    """
    if not isinstance(text, str):
        raise ValueError("Model output is not a string.")
    text = text.strip()

    # 1) Markdown fenced block
    m = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    candidates = [m.group(1).strip()] if m else []
    # 2) Whole text
    candidates.append(text)
    # 3) Outermost {...}
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        candidates.append(text[start:end + 1])

    for cand in candidates:
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("Could not parse a JSON object from model output.")


def wall_bbox(wall: Dict[str, Any], default_thickness: int = 10) -> Optional[List[float]]:
    """Return a wall's bbox, deriving one from start/end endpoints if needed."""
    if _is_valid_box(wall.get("bbox")):
        return list(wall["bbox"])
    s, e = wall.get("start"), wall.get("end")
    if _is_valid_point(s) and _is_valid_point(e):
        half = default_thickness / 2
        xmin, xmax = min(s[0], e[0]), max(s[0], e[0])
        ymin, ymax = min(s[1], e[1]), max(s[1], e[1])
        if xmax - xmin < default_thickness:
            xmin, xmax = xmin - half, xmax + half
        if ymax - ymin < default_thickness:
            ymin, ymax = ymin - half, ymax + half
        return [ymin, xmin, ymax, xmax]
    return None

In [ ]:
%%writefile src/dataset_prep.py
"""
Dataset Preparation for 2D Floor Plan Detection.

1. Generates a reproducible synthetic CAD-style floor plan dataset (image + ground-truth JSON).
2. Normalizes every coordinate to integers in [0, 1000].
3. Writes Qwen2.5-VL chat-format JSONL splits (train / val / test).

Usage (from anywhere):
    python src/dataset_prep.py --samples 120
"""

import argparse
import json
import random
from pathlib import Path
from typing import Any, Dict, List

from PIL import Image, ImageDraw, ImageFont

from schema import (
    FLOORPLAN_PROMPT, PROJECT_ROOT, ROOM_TYPES, to_target_text, validate_floorplan_json,
)

WALL_COLOR = (30, 30, 30)
WHITE = (255, 255, 255)


def _font(size: int):
    try:
        return ImageFont.load_default(size=size)  # Pillow >= 10.1
    except TypeError:
        return ImageFont.load_default()


def _norm(v: float, dim: int) -> int:
    return max(0, min(1000, int(round(v / dim * 1000))))


def _norm_box(box: List[float], w: int, h: int) -> List[int]:
    ymin, xmin, ymax, xmax = box
    return [_norm(ymin, h), _norm(xmin, w), _norm(ymax, h), _norm(xmax, w)]


def _norm_pt(pt: List[float], w: int, h: int) -> List[int]:
    return [_norm(pt[0], w), _norm(pt[1], h)]


def _draw_swing_door(draw, x0, y_wall, length, wall_half, downward=True):
    """Door opening in a horizontal wall at y_wall with a leaf and a 90deg swing arc."""
    draw.rectangle([x0, y_wall - wall_half, x0 + length, y_wall + wall_half], fill=WHITE)
    sign = 1 if downward else -1
    y_face = y_wall + sign * wall_half
    draw.line([x0, y_face, x0, y_face + sign * length], fill=(10, 10, 10), width=2)
    box = [x0 - length, y_face - length, x0 + length, y_face + length]
    draw.arc(box, start=0 if downward else 270, end=90 if downward else 360, fill=(120, 120, 120), width=1)


def _draw_swing_door_vertical(draw, x_wall, y0, length, wall_half):
    """Door opening in a vertical wall at x_wall, swinging to the right."""
    draw.rectangle([x_wall - wall_half, y0, x_wall + wall_half, y0 + length], fill=WHITE)
    x_face = x_wall + wall_half
    draw.line([x_face, y0, x_face + length, y0], fill=(10, 10, 10), width=2)
    draw.arc([x_face - length, y0 - length, x_face + length, y0 + length], start=0, end=90,
             fill=(120, 120, 120), width=1)


def _draw_sliding_door(draw, x0, y_wall, length, wall_half):
    """Sliding door: opening with two offset parallel panels."""
    draw.rectangle([x0, y_wall - wall_half, x0 + length, y_wall + wall_half], fill=WHITE)
    half = length // 2
    draw.line([x0, y_wall - 2, x0 + half + 6, y_wall - 2], fill=(10, 10, 10), width=3)
    draw.line([x0 + half - 6, y_wall + 2, x0 + length, y_wall + 2], fill=(10, 10, 10), width=3)


def generate_synthetic_floorplan(img_idx: int, output_img_dir: Path, rng: random.Random,
                                 canvas_size: int = 1000) -> Dict[str, Any]:
    """Generate one CAD-style floor plan image and its normalized ground-truth JSON."""
    W = H = canvas_size
    img = Image.new("RGB", (W, H), color=WHITE)
    draw = ImageDraw.Draw(img)

    m = rng.randint(60, 110)                # outer margin
    ext_t = rng.choice([12, 14, 16])        # exterior wall thickness
    int_t = rng.choice([6, 8, 10])          # interior wall thickness
    x0, y0, x1, y1 = m, m, W - m, H - m
    sx = rng.randint(int(W * 0.40), int(W * 0.60))   # vertical divider x
    sy = rng.randint(int(H * 0.40), int(H * 0.60))   # horizontal divider y
    ih = int_t // 2

    # ---------------- Walls ----------------
    walls_px = [
        ("wall_ext_north", "exterior", [x0, y0 + ext_t / 2], [x1, y0 + ext_t / 2], [y0, x0, y0 + ext_t, x1]),
        ("wall_ext_south", "exterior", [x0, y1 - ext_t / 2], [x1, y1 - ext_t / 2], [y1 - ext_t, x0, y1, x1]),
        ("wall_ext_west", "exterior", [x0 + ext_t / 2, y0], [x0 + ext_t / 2, y1], [y0, x0, y1, x0 + ext_t]),
        ("wall_ext_east", "exterior", [x1 - ext_t / 2, y0], [x1 - ext_t / 2, y1], [y0, x1 - ext_t, y1, x1]),
        ("wall_int_vertical", "interior", [sx, y0 + ext_t], [sx, y1 - ext_t], [y0 + ext_t, sx - ih, y1 - ext_t, sx + ih]),
        ("wall_int_horizontal", "interior", [x0 + ext_t, sy], [x1 - ext_t, sy], [sy - ih, x0 + ext_t, sy + ih, x1 - ext_t]),
    ]
    for _, _, _, _, (bymin, bxmin, bymax, bxmax) in walls_px:
        draw.rectangle([bxmin, bymin, bxmax, bymax], fill=WALL_COLOR)

    # ---------------- Rooms (random type per quadrant, labelled in the drawing) ----------------
    quadrants = [
        [y0 + ext_t, x0 + ext_t, sy - ih, sx - ih],   # top-left
        [y0 + ext_t, sx + ih, sy - ih, x1 - ext_t],   # top-right
        [sy + ih, x0 + ext_t, y1 - ext_t, sx - ih],   # bottom-left
        [sy + ih, sx + ih, y1 - ext_t, x1 - ext_t],   # bottom-right
    ]
    room_types = rng.sample(ROOM_TYPES, 4)
    tints = {"bathroom": (242, 247, 252), "kitchen": (253, 249, 242)}
    font = _font(22)
    rooms_px = []
    for i, (box, rtype) in enumerate(zip(quadrants, room_types)):
        ymin, xmin, ymax, xmax = box
        if rtype in tints:
            draw.rectangle([xmin + 2, ymin + 2, xmax - 2, ymax - 2], fill=tints[rtype])
        draw.text(((xmin + xmax) // 2, (ymin + ymax) // 2), rtype.replace("_", " ").upper(),
                  fill=(90, 90, 90), font=font, anchor="mm")
        rooms_px.append((f"room_{i + 1}", rtype, box))

    # ---------------- Doors ----------------
    doors_px = []
    pad = 8  # bbox padding around the opening
    # Entrance on the south exterior wall (left half)
    dl = rng.randint(45, 65)
    ex = rng.randint(x0 + ext_t + 30, sx - ih - dl - 30)
    draw.rectangle([ex, y1 - ext_t, ex + dl, y1], fill=WHITE)
    draw.line([ex, y1 - ext_t, ex, y1 - ext_t - dl], fill=(10, 10, 10), width=2)
    draw.arc([ex - dl, y1 - ext_t - dl, ex + dl, y1 - ext_t + dl], start=270, end=360, fill=(120, 120, 120), width=1)
    doors_px.append(("door_entrance", "entrance", [y1 - ext_t - pad, ex, y1 + pad, ex + dl]))

    # Interior door on horizontal divider, left half
    dl = rng.randint(40, 55)
    dx = rng.randint(x0 + ext_t + 25, sx - ih - dl - 25)
    _draw_swing_door(draw, dx, sy, dl, ih, downward=True)
    doors_px.append(("door_2", "interior", [sy - ih - pad, dx, sy + ih + pad, dx + dl]))

    # Door on horizontal divider, right half (interior swing or sliding)
    dl = rng.randint(45, 70)
    dx = rng.randint(sx + ih + 25, x1 - ext_t - dl - 25)
    dtype = rng.choice(["interior", "sliding"])
    if dtype == "sliding":
        _draw_sliding_door(draw, dx, sy, dl, ih)
    else:
        _draw_swing_door(draw, dx, sy, dl, ih, downward=True)
    doors_px.append(("door_3", dtype, [sy - ih - pad, dx, sy + ih + pad, dx + dl]))

    # Interior door on vertical divider (top half)
    dl = rng.randint(40, 55)
    dy = rng.randint(y0 + ext_t + 25, sy - ih - dl - 25)
    _draw_swing_door_vertical(draw, sx, dy, dl, ih)
    doors_px.append(("door_4", "interior", [dy, sx - ih - pad, dy + dl, sx + ih + pad]))

    # ---------------- Windows (2-4, on exterior walls) ----------------
    windows_px = []
    wl = rng.randint(60, 95)
    candidates = [
        ("north", rng.randint(x0 + ext_t + 20, sx - ih - wl - 20)),
        ("north", rng.randint(sx + ih + 20, x1 - ext_t - wl - 20)),
        ("west", rng.randint(sy + ih + 20, y1 - ext_t - wl - 20)),
        ("east", rng.randint(y0 + ext_t + 20, sy - ih - wl - 20)),
    ]
    chosen = rng.sample(candidates, rng.randint(2, 4))
    for i, (side, pos) in enumerate(chosen):
        if side == "north":
            draw.rectangle([pos, y0, pos + wl, y0 + ext_t], fill=WHITE, outline=(40, 40, 40), width=2)
            draw.line([pos, y0 + ext_t // 2, pos + wl, y0 + ext_t // 2], fill=(70, 130, 180), width=2)
            box = [y0 - 4, pos, y0 + ext_t + 4, pos + wl]
        elif side == "west":
            draw.rectangle([x0, pos, x0 + ext_t, pos + wl], fill=WHITE, outline=(40, 40, 40), width=2)
            draw.line([x0 + ext_t // 2, pos, x0 + ext_t // 2, pos + wl], fill=(70, 130, 180), width=2)
            box = [pos, x0 - 4, pos + wl, x0 + ext_t + 4]
        else:  # east
            draw.rectangle([x1 - ext_t, pos, x1, pos + wl], fill=WHITE, outline=(40, 40, 40), width=2)
            draw.line([x1 - ext_t // 2, pos, x1 - ext_t // 2, pos + wl], fill=(70, 130, 180), width=2)
            box = [pos, x1 - ext_t - 4, pos + wl, x1 + 4]
        windows_px.append((f"window_{i + 1}", box))

    # ---------------- Save image ----------------
    img_filename = f"floorplan_{img_idx:04d}.png"
    img_path = output_img_dir / img_filename
    img.save(img_path)

    # ---------------- Normalized ground truth ----------------
    gt = {
        "image_dimensions": {"width": W, "height": H},
        "rooms": [{"id": rid, "type": t, "bbox": _norm_box(b, W, H)} for rid, t, b in rooms_px],
        "walls": [
            {"id": wid, "type": t, "start": _norm_pt(s, W, H), "end": _norm_pt(e, W, H), "bbox": _norm_box(b, W, H)}
            for wid, t, s, e, b in walls_px
        ],
        "doors": [{"id": did, "type": t, "bbox": _norm_box(b, W, H)} for did, t, b in doors_px],
        "windows": [{"id": wid, "bbox": _norm_box(b, W, H)} for wid, b in windows_px],
    }
    ok, errs = validate_floorplan_json(gt)
    assert ok, f"Generated invalid ground truth for {img_filename}: {errs}"

    try:
        rel = img_path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        rel = img_path.resolve().as_posix()
    return {"id": f"floorplan_{img_idx:04d}", "image_path": rel, "abs_image_path": str(img_path), "ground_truth": gt}


def to_chat_record(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "id": item["id"],
        "image": item["image_path"],
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": item["image_path"]},
                {"type": "text", "text": FLOORPLAN_PROMPT},
            ]},
            # Content as a list (same structure as the user turn) keeps HF datasets / Arrow happy.
            {"role": "assistant", "content": [{"type": "text", "text": to_target_text(item["ground_truth"])}]},
        ],
    }


def create_dataset_splits(num_samples: int = 120, output_dir: str = None, train_ratio: float = 0.8,
                          val_ratio: float = 0.1, seed: int = 42, canvas_size: int = 1000):
    base_dir = Path(output_dir) if output_dir else PROJECT_ROOT / "data"
    img_dir, sample_dir, proc_dir = base_dir / "raw" / "images", base_dir / "samples", base_dir / "processed"
    for d in (img_dir, sample_dir, proc_dir):
        d.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)  # fully reproducible
    print(f"Generating {num_samples} synthetic floor plans (seed={seed})...")
    items = [generate_synthetic_floorplan(i, img_dir, rng, canvas_size) for i in range(1, num_samples + 1)]

    rng.shuffle(items)
    n_train = int(num_samples * train_ratio)
    n_val = int(num_samples * val_ratio)
    splits = {"train": items[:n_train], "val": items[n_train:n_train + n_val], "test": items[n_train + n_val:]}

    # Sample deliverables are taken from the TEST split (never seen in training)
    for i, item in enumerate(splits["test"][:5], start=1):
        with open(sample_dir / f"sample_{i:02d}.json", "w", encoding="utf-8") as f:
            json.dump(item["ground_truth"], f, indent=2)
        Image.open(item["abs_image_path"]).save(sample_dir / f"sample_{i:02d}.png")

    for name, split_items in splits.items():
        path = proc_dir / f"{name}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for item in split_items:
                f.write(json.dumps(to_chat_record(item), ensure_ascii=False) + "\n")
        print(f"Wrote {len(split_items):>4} samples to {path}")

    print(f"\nDone. Images: {img_dir}\nSamples (from test split): {sample_dir}\nJSONL: {proc_dir}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Floor Plan Dataset Preprocessing")
    parser.add_argument("--samples", type=int, default=120, help="Number of synthetic samples to generate")
    parser.add_argument("--outdir", type=str, default=None, help="Output directory (default: <project>/data)")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    create_dataset_splits(num_samples=args.samples, output_dir=args.outdir, seed=args.seed)

In [ ]:
%%writefile src/train.py
"""
QLoRA fine-tuning of Qwen2.5-VL for 2D floor plan element detection.

Fixes vs. the original version:
  * nested chat messages are stored as JSON strings (Arrow cannot hold the mixed nested schema and
    would inject `image: None` keys that break qwen_vl_utils),
  * remove_unused_columns=False (otherwise Trainer silently drops the `messages` column -> empty batches),
  * `eval_strategy` instead of the removed `evaluation_strategy` argument (and v5-safe warmup),
  * LoRA restricted to the language model (the ViT's MLP also has gate/up/down_proj layers),
  * loss is computed ONLY on the assistant answer (prompt, image and padding tokens are masked to -100),
  * image paths are resolved relative to the project root,
  * works on CPU for smoke tests (no bitsandbytes / bf16 calls without CUDA).

Usage:
    python src/train.py --epochs 3
"""

import argparse
import inspect
import json
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

from schema import PROJECT_ROOT, resolve_path

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28
ASSISTANT_HEADER = "<|im_start|>assistant\n"
LORA_TARGET_REGEX = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"


def load_dataset_from_jsonl(jsonl_path: str) -> Dataset:
    """Loads chat JSONL, resolves image paths, stores messages as a JSON string column."""
    jsonl_path = Path(resolve_path(jsonl_path))
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            for msg in rec["messages"]:
                if isinstance(msg["content"], list):
                    for part in msg["content"]:
                        if part.get("type") == "image":
                            part["image"] = resolve_path(part["image"], base=PROJECT_ROOT)
            rows.append({"id": rec["id"], "messages_json": json.dumps(rec["messages"], ensure_ascii=False)})
    if not rows:
        raise ValueError(f"No records found in {jsonl_path}")
    return Dataset.from_list(rows)


class FloorplanCollator:
    """Tokenizes multimodal conversations and masks everything except the assistant answer."""

    def __init__(self, processor):
        self.processor = processor
        self.processor.tokenizer.padding_side = "right"
        self.header_ids = processor.tokenizer.encode(ASSISTANT_HEADER, add_special_tokens=False)
        self.pad_id = processor.tokenizer.pad_token_id

    def _answer_start(self, ids):
        """Index right after the LAST '<|im_start|>assistant\\n' header."""
        n = len(self.header_ids)
        for i in range(len(ids) - n, -1, -1):
            if ids[i:i + n] == self.header_ids:
                return i + n
        return None

    def __call__(self, batch):
        messages_list = [json.loads(item["messages_json"]) for item in batch]
        texts = [self.processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                 for m in messages_list]
        image_inputs, video_inputs = process_vision_info(messages_list)
        inputs = self.processor(text=texts, images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt")

        labels = inputs["input_ids"].clone()
        labels[inputs["attention_mask"] == 0] = -100
        for i in range(labels.size(0)):
            start = self._answer_start(inputs["input_ids"][i].tolist())
            if start is None:
                raise ValueError("Assistant header not found - check the chat template / data format.")
            labels[i, :start] = -100
        if (labels != -100).sum() == 0:
            raise ValueError("No trainable answer tokens in batch.")
        inputs["labels"] = labels
        return inputs


def train(args):
    use_cuda = torch.cuda.is_available()
    bf16 = use_cuda and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if bf16 else (torch.float16 if use_cuda else torch.float32)

    output_dir = Path(args.output_dir)
    if not output_dir.is_absolute():
        output_dir = PROJECT_ROOT / output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading base model: {args.model_id} (cuda={use_cuda}, dtype={compute_dtype})")
    quant_config = None
    if use_cuda and not args.no_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True,
        )

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        args.model_id,
        quantization_config=quant_config,
        torch_dtype=compute_dtype,
        device_map="auto" if use_cuda else None,
    )
    model.config.use_cache = False  # required with gradient checkpointing

    processor = AutoProcessor.from_pretrained(args.model_id, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

    if quant_config is not None:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=args.lora_r, lora_alpha=args.lora_alpha, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        # LoRA on the language model only: the vision tower's MLP also uses gate/up/down_proj names,
        # so a plain name list would silently adapt the ViT as well.
        target_modules=LORA_TARGET_REGEX,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("Loading datasets...")
    train_ds = load_dataset_from_jsonl(args.train_file)
    val_ds = load_dataset_from_jsonl(args.val_file)
    print(f"  train={len(train_ds)}  val={len(val_ds)}")

    # transformers v5 removed `warmup_ratio` (warmup_steps accepts a float ratio instead)
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    warmup = {"warmup_ratio": 0.05} if "warmup_ratio" in ta_params else {"warmup_steps": 0.05}

    training_args = TrainingArguments(
        **warmup,
        output_dir=str(output_dir),
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        num_train_epochs=args.epochs,
        learning_rate=args.learning_rate,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=bf16,
        fp16=use_cuda and not bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,   # keep `messages_json` for the collator
        dataloader_pin_memory=use_cuda,
        prediction_loss_only=True,
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=FloorplanCollator(processor),
    )

    print("Starting fine-tuning...")
    trainer.train()

    adapter_path = output_dir / "final_adapter"
    trainer.model.save_pretrained(adapter_path)
    processor.save_pretrained(adapter_path)
    print(f"Fine-tuning complete! LoRA adapter saved to {adapter_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Fine-tune Qwen2.5-VL for 2D Floor Plan Detection")
    parser.add_argument("--model_id", type=str, default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--train_file", type=str, default="data/processed/train.jsonl")
    parser.add_argument("--val_file", type=str, default="data/processed/val.jsonl")
    parser.add_argument("--output_dir", type=str, default="checkpoints/qwen_floorplan_lora")
    parser.add_argument("--epochs", type=float, default=3)
    parser.add_argument("--batch_size", type=int, default=1)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=8)
    parser.add_argument("--learning_rate", type=float, default=2e-4)
    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--no_4bit", action="store_true", help="Disable 4-bit quantization on GPU")
    args = parser.parse_args()
    train(args)

In [ ]:
%%writefile src/inference.py
"""
Inference for 2D Floor Plan Detection.

Engines:
  * hf     - Hugging Face Transformers + (optional) LoRA adapter
  * ollama - local Ollama daemon (base64 image via /api/generate)

Usage:
    python src/inference.py --image data/samples/sample_01.png --output_json out.json
    python src/inference.py --engine ollama --model floorplan-vlm --image data/samples/sample_01.png
"""

import argparse
import base64
import json
import os
from pathlib import Path
from typing import Optional

from schema import FLOORPLAN_PROMPT, PROJECT_ROOT, extract_json_from_response, resolve_path, validate_floorplan_json

DEFAULT_HF_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
DEFAULT_OLLAMA_MODEL = "qwen2.5vl:3b"
DEFAULT_ADAPTER = "checkpoints/qwen_floorplan_lora/final_adapter"
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28


class HFFloorplanDetector:
    """Loads the model once; call .predict(image_path) many times (used by evaluate.py too)."""

    def __init__(self, model_id: str = DEFAULT_HF_MODEL, adapter_path: Optional[str] = DEFAULT_ADAPTER,
                 load_in_4bit: bool = False, max_new_tokens: int = 2048):
        import torch
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

        self.torch = torch
        self.max_new_tokens = max_new_tokens
        use_cuda = torch.cuda.is_available()
        dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if use_cuda else torch.float32

        kwargs = {"torch_dtype": dtype, "device_map": "auto" if use_cuda else None}
        if load_in_4bit and use_cuda:
            from transformers import BitsAndBytesConfig
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
                bnb_4bit_use_double_quant=True)

        print(f"Loading model {model_id} ...")
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **kwargs)

        processor_src = model_id
        if adapter_path and str(adapter_path).lower() in ("none", "base", ""):
            print("Running the BASE model without a LoRA adapter.")
            adapter_path = None
        adapter_path = resolve_path(adapter_path) if adapter_path else None
        if adapter_path and os.path.isdir(adapter_path):
            from peft import PeftModel
            print(f"Loading LoRA adapter from {adapter_path} ...")
            self.model = PeftModel.from_pretrained(self.model, adapter_path)
            if os.path.exists(os.path.join(adapter_path, "preprocessor_config.json")):
                processor_src = adapter_path  # same image resolution settings as training
        elif adapter_path:
            print(f"WARNING: adapter '{adapter_path}' not found -> running the BASE model (not fine-tuned).")

        self.processor = AutoProcessor.from_pretrained(processor_src, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
        self.model.eval()

    def predict(self, image_path: str) -> str:
        from qwen_vl_utils import process_vision_info

        image_path = resolve_path(image_path, base=PROJECT_ROOT)
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": FLOORPLAN_PROMPT},
        ]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)

        with self.torch.inference_mode():
            generated = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens,
                                            do_sample=False, use_cache=True)
        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
        return self.processor.batch_decode(trimmed, skip_special_tokens=True,
                                           clean_up_tokenization_spaces=False)[0]


def run_ollama_inference(image_path: str, model_tag: str = DEFAULT_OLLAMA_MODEL,
                         host: str = "http://localhost:11434", timeout: int = 600) -> str:
    import requests

    image_path = resolve_path(image_path, base=PROJECT_ROOT)
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")

    payload = {
        "model": model_tag,
        "prompt": FLOORPLAN_PROMPT,
        "images": [img_b64],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 2048},
    }
    url = f"{host.rstrip('/')}/api/generate"
    print(f"Sending request to Ollama: {url} (model: {model_tag}) ...")
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.ConnectionError as e:
        raise SystemExit(f"Cannot reach Ollama at {host}. Is `ollama serve` running?") from e
    if resp.status_code == 404:
        raise SystemExit(f"Ollama model '{model_tag}' not found. Run `ollama pull {model_tag}` or `ollama create`.")
    resp.raise_for_status()
    return resp.json().get("response", "")


def main():
    parser = argparse.ArgumentParser(description="Floor Plan Detection Inference")
    parser.add_argument("--image", type=str, required=True, help="Path to input floor plan image")
    parser.add_argument("--engine", choices=["hf", "ollama"], default="hf")
    parser.add_argument("--model", type=str, default=None,
                        help=f"HF model id (default {DEFAULT_HF_MODEL}) or Ollama tag (default {DEFAULT_OLLAMA_MODEL})")
    parser.add_argument("--adapter", type=str, default=DEFAULT_ADAPTER)
    parser.add_argument("--host", type=str, default="http://localhost:11434")
    parser.add_argument("--load_in_4bit", action="store_true")
    parser.add_argument("--output_json", type=str, default=None, help="Path to save parsed JSON")
    parser.add_argument("--visualize", action="store_true", help="Also save 2D overlay + 3D preview")
    args = parser.parse_args()

    if not Path(resolve_path(args.image)).exists():
        raise SystemExit(f"Image not found: {args.image}")

    if args.engine == "hf":
        detector = HFFloorplanDetector(args.model or DEFAULT_HF_MODEL, args.adapter, args.load_in_4bit)
        raw_output = detector.predict(args.image)
    else:
        raw_output = run_ollama_inference(args.image, args.model or DEFAULT_OLLAMA_MODEL, args.host)

    print("\n--- Raw Model Output ---")
    print(raw_output)

    try:
        parsed = extract_json_from_response(raw_output)
    except ValueError as e:
        print(f"\nFailed to parse JSON from output: {e}")
        raise SystemExit(1)

    is_valid, errors = validate_floorplan_json(parsed)
    print("\n--- JSON Validation ---")
    print("Schema valid:", is_valid)
    for err in errors:
        print("  -", err)

    if args.output_json:
        Path(args.output_json).parent.mkdir(parents=True, exist_ok=True)
        with open(args.output_json, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=2)
        print(f"Saved structured JSON to {args.output_json}")

    if args.visualize:
        from visualize import draw_2d_detections, render_3d_extrusion
        stem = Path(args.output_json or Path(args.image).stem + "_pred.json").with_suffix("")
        draw_2d_detections(resolve_path(args.image), parsed, f"{stem}_overlay.png")
        render_3d_extrusion(parsed, f"{stem}_3d.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/evaluate.py
"""
Evaluation for 2D Floor Plan Detection.

Metrics (per class: rooms, walls, doors, windows):
  * JSON parse rate and schema validity rate
  * Precision / Recall / F1 at IoU 0.50 and 0.75 (optimal-greedy one-to-one matching)
  * mean F1 over IoU thresholds 0.50:0.95 (COCO-style threshold sweep)
  * mean IoU of matched pairs
  * type (sub-class) accuracy of matched pairs (e.g. bedroom vs kitchen)

NOTE: the model emits no confidence scores, so a true mAP (area under a ranked PR curve) is not
defined. F1 at fixed IoU thresholds is reported instead.

Modes:
  python src/evaluate.py --run_inference                    # run the fine-tuned model on the test split
  python src/evaluate.py --predictions data/predictions_test.jsonl   # score saved predictions
  python src/evaluate.py --oracle                           # pipeline sanity check (GT vs GT == 1.0)
"""

import argparse
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np

from schema import CATEGORIES, PROJECT_ROOT, extract_json_from_response, resolve_path, \
    validate_floorplan_json, wall_bbox

IOU_SWEEP = [round(x, 2) for x in np.arange(0.50, 0.96, 0.05)]


def calculate_iou(box_a: List[float], box_b: List[float]) -> float:
    """IoU for boxes in [ymin, xmin, ymax, xmax] format."""
    y_a, x_a = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    y_b, x_b = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    inter = max(0.0, y_b - y_a) * max(0.0, x_b - x_a)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _box_of(item: Any, category: str):
    if not isinstance(item, dict):
        return None
    box = wall_bbox(item) if category == "walls" else item.get("bbox")
    if isinstance(box, (list, tuple)) and len(box) == 4 and all(
            isinstance(v, (int, float)) and not isinstance(v, bool) for v in box):
        return [float(v) for v in box]
    return None


def match_boxes(preds: List[Dict], gts: List[Dict], category: str, iou_threshold: float = 0.5
                ) -> Tuple[int, int, int, List[float], List[bool]]:
    """
    One-to-one matching: all (pred, gt) pairs sorted by IoU, taken greedily (highest first).
    Returns (TP, FP, FN, matched IoUs, type-correct flags for matched pairs).
    """
    p_boxes = [_box_of(p, category) for p in preds]
    g_boxes = [_box_of(g, category) for g in gts]
    pairs = []
    for i, pb in enumerate(p_boxes):
        if pb is None:
            continue
        for j, gb in enumerate(g_boxes):
            if gb is None:
                continue
            iou = calculate_iou(pb, gb)
            if iou >= iou_threshold:
                pairs.append((iou, i, j))
    pairs.sort(reverse=True)

    used_p, used_g, ious, type_ok = set(), set(), [], []
    for iou, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        ious.append(iou)
        gt_type = gts[j].get("type") if isinstance(gts[j], dict) else None
        if gt_type is not None:
            type_ok.append(preds[i].get("type") == gt_type)

    tp = len(used_p)
    fp = len(preds) - tp
    fn = len(gts) - tp
    return tp, fp, fn, ious, type_ok


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def evaluate_predictions(gt_by_id: Dict[str, Dict], pred_by_id: Dict[str, Any]) -> Dict:
    """gt_by_id: id -> ground-truth dict.  pred_by_id: id -> raw text or parsed dict (missing = failure)."""
    n = len(gt_by_id)
    parsed_ok, schema_ok = 0, 0
    parsed_preds: Dict[str, Dict] = {}

    for sid in gt_by_id:
        raw = pred_by_id.get(sid)
        try:
            pred = extract_json_from_response(raw) if isinstance(raw, str) else raw
            if not isinstance(pred, dict):
                raise ValueError("not a dict")
            parsed_ok += 1
            if validate_floorplan_json(pred)[0]:
                schema_ok += 1
            parsed_preds[sid] = pred
        except Exception:
            parsed_preds[sid] = {}  # unparseable -> every GT element is a false negative

    summary: Dict[str, Any] = {
        "total_test_samples": n,
        "json_parse_rate_percent": round(100 * parsed_ok / n, 2) if n else 0.0,
        "schema_validity_rate_percent": round(100 * schema_ok / n, 2) if n else 0.0,
        "per_class_metrics": {},
    }

    macro = {k: [] for k in ["precision@0.50", "recall@0.50", "f1@0.50", "f1@0.75", "mean_f1@[.50:.95]"]}
    for c in CATEGORIES:
        sweep_f1 = []
        stats_50 = stats_75 = None
        all_ious, all_types = [], []
        for thr in IOU_SWEEP:
            tp = fp = fn = 0
            for sid, gt in gt_by_id.items():
                pred_list = parsed_preds[sid].get(c, [])
                pred_list = pred_list if isinstance(pred_list, list) else []
                t, f_p, f_n, ious, types = match_boxes(pred_list, gt.get(c, []), c, thr)
                tp, fp, fn = tp + t, fp + f_p, fn + f_n
                if thr == 0.5:
                    all_ious += ious
                    all_types += types
            prf = _prf(tp, fp, fn)
            sweep_f1.append(prf[2])
            if thr == 0.5:
                stats_50 = (tp, fp, fn, *prf)
            if thr == 0.75:
                stats_75 = prf
        tp, fp, fn, p, r, f = stats_50
        entry = {
            "tp@0.50": tp, "fp@0.50": fp, "fn@0.50": fn,
            "precision@0.50": round(p, 4), "recall@0.50": round(r, 4), "f1@0.50": round(f, 4),
            "f1@0.75": round(stats_75[2], 4),
            "mean_f1@[.50:.95]": round(float(np.mean(sweep_f1)), 4),
            "mean_matched_iou": round(float(np.mean(all_ious)), 4) if all_ious else 0.0,
        }
        if all_types:
            entry["type_accuracy"] = round(sum(all_types) / len(all_types), 4)
        summary["per_class_metrics"][c] = entry
        for k in macro:
            macro[k].append(entry[k])

    summary["macro"] = {k: round(float(np.mean(v)), 4) for k, v in macro.items()}
    return summary


def load_ground_truth(test_jsonl: Path) -> Tuple[Dict[str, Dict], Dict[str, str]]:
    gts, images = {}, {}
    with open(test_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            content = rec["messages"][-1]["content"]
            text = content if isinstance(content, str) else "".join(
                p.get("text", "") for p in content if isinstance(p, dict))
            gts[rec["id"]] = extract_json_from_response(text)
            images[rec["id"]] = rec["image"]
    return gts, images


def load_predictions(path: Path) -> Dict[str, Any]:
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                preds[rec["id"]] = rec.get("prediction")
    return preds


def print_report(m: Dict, title: str):
    print(f"\n==================== {title} ====================")
    print(f"Samples: {m['total_test_samples']}   JSON parse rate: {m['json_parse_rate_percent']}%   "
          f"Schema valid: {m['schema_validity_rate_percent']}%")
    print(f"{'class':<9}{'P@.5':>8}{'R@.5':>8}{'F1@.5':>8}{'F1@.75':>8}{'F1@.5:.95':>11}{'mIoU':>8}{'typeAcc':>9}")
    for c, v in m["per_class_metrics"].items():
        print(f"{c:<9}{v['precision@0.50']:>8}{v['recall@0.50']:>8}{v['f1@0.50']:>8}{v['f1@0.75']:>8}"
              f"{v['mean_f1@[.50:.95]']:>11}{v['mean_matched_iou']:>8}{str(v.get('type_accuracy', '-')):>9}")
    mm = m["macro"]
    print(f"{'MACRO':<9}{mm['precision@0.50']:>8}{mm['recall@0.50']:>8}{mm['f1@0.50']:>8}{mm['f1@0.75']:>8}"
          f"{mm['mean_f1@[.50:.95]']:>11}")
    print("=" * (42 + len(title)))


def main():
    ap = argparse.ArgumentParser(description="Evaluate floor plan detection on the test split")
    ap.add_argument("--test_file", default="data/processed/test.jsonl")
    ap.add_argument("--predictions", default=None, help="JSONL with {id, prediction} per line")
    ap.add_argument("--run_inference", action="store_true", help="Run the HF model on the test split first")
    ap.add_argument("--model", default="Qwen/Qwen2.5-VL-3B-Instruct")
    ap.add_argument("--adapter", default="checkpoints/qwen_floorplan_lora/final_adapter")
    ap.add_argument("--load_in_4bit", action="store_true")
    ap.add_argument("--limit", type=int, default=None, help="Evaluate only the first N test samples")
    ap.add_argument("--oracle", action="store_true", help="Score GT against itself (pipeline sanity check)")
    ap.add_argument("--out", default=None, help="Where to write the metrics JSON")
    args = ap.parse_args()

    test_jsonl = Path(resolve_path(args.test_file))
    if not test_jsonl.exists():
        raise SystemExit(f"{test_jsonl} not found. Run `python src/dataset_prep.py` first.")
    gts, images = load_ground_truth(test_jsonl)
    if args.limit:
        gts = dict(list(gts.items())[:args.limit])

    if args.oracle:
        preds, mode = dict(gts), "oracle_sanity_check (ground truth scored against itself - NOT model performance)"
        default_out = PROJECT_ROOT / "data" / "evaluation_oracle_sanity_check.json"
    elif args.predictions:
        preds, mode = load_predictions(Path(resolve_path(args.predictions))), f"predictions:{args.predictions}"
        default_out = PROJECT_ROOT / "data" / "evaluation_results.json"
    elif args.run_inference:
        from inference import HFFloorplanDetector
        det = HFFloorplanDetector(args.model, args.adapter, args.load_in_4bit)
        pred_path = PROJECT_ROOT / "data" / "predictions_test.jsonl"
        preds = {}
        with open(pred_path, "w", encoding="utf-8") as f:
            for k, sid in enumerate(gts, 1):
                print(f"[{k}/{len(gts)}] {sid}")
                raw = det.predict(images[sid])
                preds[sid] = raw
                f.write(json.dumps({"id": sid, "prediction": raw}, ensure_ascii=False) + "\n")
        print(f"Saved raw predictions to {pred_path}")
        mode = f"model:{args.model} adapter:{args.adapter}"
        default_out = PROJECT_ROOT / "data" / "evaluation_results.json"
    else:
        ap.print_help()
        raise SystemExit("\nChoose one of --run_inference, --predictions FILE or --oracle.")

    metrics = evaluate_predictions(gts, preds)
    metrics = {"mode": mode, **metrics}
    out = Path(args.out) if args.out else default_out
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print_report(metrics, "Evaluation" if not args.oracle else "ORACLE sanity check")
    print(f"Saved metrics to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/visualize.py
"""
Visualization for 2D Floor Plan Detection + downstream 2D-to-3D bridge.

1. 2D overlay of detected boxes and labels on the floor plan image.
2. 3D extrusion: walls are extruded to `wall_height`, door/window openings are cut out of the walls
   (lintels above doors, sills + lintels around windows), rooms become floor slabs.

Usage:
    python src/visualize.py --image data/samples/sample_01.png --json data/samples/sample_01.json
"""

import argparse
import json
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")  # headless-safe
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image, ImageDraw, ImageFont

from schema import PROJECT_ROOT, resolve_path, wall_bbox

CLASS_COLORS = {
    "rooms": (46, 204, 113),
    "walls": (52, 73, 94),
    "doors": (230, 126, 34),
    "windows": (52, 152, 219),
}
ROOM_COLORS = {
    "living_room": "#f6d7a7", "bedroom": "#c9e4de", "kitchen": "#faedcb",
    "bathroom": "#c6def1", "hallway": "#e2e2df", "dining_room": "#f2c6de",
}
DOOR_HEIGHT, SILL_HEIGHT, WINDOW_TOP = 2.1, 0.9, 2.1


def _font(size=14):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def _item_box(item: Dict, category: str):
    box = wall_bbox(item) if category == "walls" else item.get("bbox")
    if isinstance(box, (list, tuple)) and len(box) == 4:
        try:
            return [float(v) for v in box]
        except (TypeError, ValueError):
            return None
    return None


def draw_2d_detections(img_path: str, json_data: dict, output_path: str, coord_max: int = 1000):
    """Overlay boxes + labels (coordinates normalized to 0..coord_max) on the image."""
    img = Image.open(resolve_path(img_path)).convert("RGBA")
    overlay = Image.new("RGBA", img.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(overlay)
    w, h = img.size
    font = _font(14)

    # walls first so smaller elements stay visible on top
    for category in ["walls", "rooms", "doors", "windows"]:
        color = CLASS_COLORS[category]
        for item in json_data.get(category, []) or []:
            if not isinstance(item, dict):
                continue
            box = _item_box(item, category)
            if box is None:
                continue
            ymin, xmin, ymax, xmax = box
            x0, x1 = int(xmin / coord_max * w), int(xmax / coord_max * w)
            y0, y1 = int(ymin / coord_max * h), int(ymax / coord_max * h)
            fill_alpha = 25 if category in ("rooms", "walls") else 70
            draw.rectangle([x0, y0, x1, y1], outline=(*color, 255), fill=(*color, fill_alpha),
                           width=1 if category == "walls" else 3)
            if category == "walls":
                continue
            label = str(item.get("type", category[:-1]))
            tw = int(draw.textlength(label, font=font)) + 6
            ty = y0 - 18 if y0 >= 18 else y0 + 2       # keep the tag inside the image
            draw.rectangle([x0, ty, x0 + tw, ty + 17], fill=(*color, 230))
            draw.text((x0 + 3, ty + 1), label, fill=(255, 255, 255, 255), font=font)

    Image.alpha_composite(img, overlay).convert("RGB").save(output_path)
    print(f"Saved 2D detection overlay to {output_path}")


def _box_faces(x0, x1, y0, y1, z0, z1):
    return [
        [(x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0)],
        [(x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1)],
        [(x0, y0, z0), (x1, y0, z0), (x1, y0, z1), (x0, y0, z1)],
        [(x0, y1, z0), (x1, y1, z0), (x1, y1, z1), (x0, y1, z1)],
        [(x0, y0, z0), (x0, y1, z0), (x0, y1, z1), (x0, y0, z1)],
        [(x1, y0, z0), (x1, y1, z0), (x1, y1, z1), (x1, y0, z1)],
    ]


def _overlap(a0, a1, b0, b1):
    return max(a0, b0), min(a1, b1)


def build_wall_segments(walls: List[List[float]], openings: List[Tuple[List[float], str]],
                        wall_height: float) -> List[Tuple[float, float, float, float, float, float]]:
    """
    Split every wall box (meters, x0,x1,y0,y1) along its long axis around the openings that intersect it.
    Returns solid boxes (x0, x1, y0, y1, z0, z1).
    """
    solids = []
    for (x0, x1, y0, y1) in walls:
        horizontal = (x1 - x0) >= (y1 - y0)
        lo, hi = (x0, x1) if horizontal else (y0, y1)
        cuts = []
        for (ox0, ox1, oy0, oy1), kind in openings:
            # opening must intersect the wall's cross-section
            cx0, cx1 = _overlap(x0, x1, ox0, ox1)
            cy0, cy1 = _overlap(y0, y1, oy0, oy1)
            if cx1 <= cx0 or cy1 <= cy0:
                continue
            a, b = (cx0, cx1) if horizontal else (cy0, cy1)
            cuts.append((a, b, kind))
        cuts.sort()

        def seg(a, b, z0, z1):
            if b - a <= 1e-6 or z1 <= z0:
                return
            if horizontal:
                solids.append((a, b, y0, y1, z0, z1))
            else:
                solids.append((x0, x1, a, b, z0, z1))

        cursor = lo
        for a, b, kind in cuts:
            a, b = max(a, cursor), max(b, cursor)
            seg(cursor, a, 0, wall_height)                       # full-height wall before the opening
            if kind == "door":
                seg(a, b, DOOR_HEIGHT, wall_height)              # lintel
            else:
                seg(a, b, 0, SILL_HEIGHT)                        # sill
                seg(a, b, WINDOW_TOP, wall_height)               # lintel
            cursor = max(cursor, b)
        seg(cursor, hi, 0, wall_height)
    return solids


def render_3d_extrusion(json_data: dict, output_path: str, wall_height: float = 2.8,
                        plan_size_m: float = 10.0, coord_max: int = 1000):
    """Extrude walls, cut door/window openings, draw room slabs. Plan is scaled to plan_size_m meters."""
    s = plan_size_m / coord_max

    def to_m(box):  # [ymin, xmin, ymax, xmax] -> (x0, x1, y0, y1) with Y flipped (image y points down)
        ymin, xmin, ymax, xmax = box
        return xmin * s, xmax * s, plan_size_m - ymax * s, plan_size_m - ymin * s

    fig = plt.figure(figsize=(11, 9))
    ax = fig.add_subplot(111, projection="3d")

    for r in json_data.get("rooms", []) or []:
        box = _item_box(r, "rooms") if isinstance(r, dict) else None
        if box is None:
            continue
        x0, x1, y0, y1 = to_m(box)
        ax.add_collection3d(Poly3DCollection(
            [[(x0, y0, 0), (x1, y0, 0), (x1, y1, 0), (x0, y1, 0)]], alpha=0.55,
            facecolor=ROOM_COLORS.get(r.get("type"), "#dddddd"), edgecolor="#7f8c8d"))
        ax.text((x0 + x1) / 2, (y0 + y1) / 2, 0.02, str(r.get("type", "")).replace("_", " "),
                fontsize=8, ha="center", color="#2c3e50")

    openings = []
    for cat, kind in (("doors", "door"), ("windows", "window")):
        for o in json_data.get(cat, []) or []:
            box = _item_box(o, cat) if isinstance(o, dict) else None
            if box is not None:
                openings.append((to_m(box), kind))

    for w in json_data.get("walls", []) or []:
        box = _item_box(w, "walls") if isinstance(w, dict) else None
        if box is None:
            continue
        color = "#2c3e50" if w.get("type") == "exterior" else "#95a5a6"
        for (x0, x1, y0, y1, z0, z1) in build_wall_segments([to_m(box)], openings, wall_height):
            ax.add_collection3d(Poly3DCollection(_box_faces(x0, x1, y0, y1, z0, z1), alpha=0.9,
                                                 facecolor=color, edgecolor="#1a252f", linewidths=0.3))

    # glass panes in window openings
    for (x0, x1, y0, y1), kind in openings:
        if kind == "window":
            xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
            if (x1 - x0) >= (y1 - y0):
                pane = [(x0, ym, SILL_HEIGHT), (x1, ym, SILL_HEIGHT), (x1, ym, WINDOW_TOP), (x0, ym, WINDOW_TOP)]
            else:
                pane = [(xm, y0, SILL_HEIGHT), (xm, y1, SILL_HEIGHT), (xm, y1, WINDOW_TOP), (xm, y0, WINDOW_TOP)]
            ax.add_collection3d(Poly3DCollection([pane], alpha=0.35, facecolor="#3498db", edgecolor="#2980b9"))

    ax.set_xlim(0, plan_size_m)
    ax.set_ylim(0, plan_size_m)
    ax.set_zlim(0, wall_height + 0.5)
    try:
        ax.set_box_aspect((plan_size_m, plan_size_m, (wall_height + 0.5) * 1.5))
    except AttributeError:
        pass
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Height (m)")
    ax.view_init(elev=38, azim=-60)
    plt.title("Downstream 2D-to-3D Extruded Model (doors/windows cut into walls)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close(fig)
    print(f"Saved 3D extrusion preview to {output_path}")


def main():
    parser = argparse.ArgumentParser(description="Floor Plan Detection & 3D Bridge Visualizer")
    parser.add_argument("--image", type=str, default=str(PROJECT_ROOT / "data/samples/sample_01.png"))
    parser.add_argument("--json", type=str, default=str(PROJECT_ROOT / "data/samples/sample_01.json"))
    parser.add_argument("--out2d", type=str, default=str(PROJECT_ROOT / "data/samples/detection_overlay_01.png"))
    parser.add_argument("--out3d", type=str, default=str(PROJECT_ROOT / "data/samples/extruded_3d_preview_01.png"))
    parser.add_argument("--wall_height", type=float, default=2.8)
    args = parser.parse_args()

    with open(resolve_path(args.json), "r", encoding="utf-8") as f:
        data = json.load(f)
    for p in (args.out2d, args.out3d):
        Path(p).parent.mkdir(parents=True, exist_ok=True)
    draw_2d_detections(args.image, data, args.out2d)
    render_3d_extrusion(data, args.out3d, wall_height=args.wall_height)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/export_to_ollama.py
"""
Merge the fine-tuned LoRA adapter into the base model and generate an Ollama Modelfile.

Usage:
    python src/export_to_ollama.py --merge                     # merge + Modelfile pointing at merged weights
    ollama create floorplan-vlm -f Modelfile
    python src/inference.py --engine ollama --model floorplan-vlm --image data/samples/sample_01.png

Notes:
  * Merging must be done on a NON-quantized (fp16/bf16) base model; the 4-bit weights used during
    QLoRA training cannot be merged into directly.
  * Ollama can only import Safetensors for the architectures its converter supports. If
    `ollama create` rejects Qwen2_5_VLForConditionalGeneration in your Ollama version, convert the merged
    folder to GGUF with llama.cpp (text model + vision projector), then point FROM at the GGUF file(s):
        python llama.cpp/convert_hf_to_gguf.py checkpoints/merged_fp16 --outfile floorplan.gguf
        python llama.cpp/convert_hf_to_gguf.py checkpoints/merged_fp16 --mmproj --outfile floorplan-mmproj.gguf
  * The HF engine (`python src/inference.py`) is the reference path and always works.
"""

import argparse
from pathlib import Path

from schema import PROJECT_ROOT, resolve_path

# No SYSTEM override: training used Qwen's default system prompt, and the task prompt is sent by
# inference.py as the user message - changing either at serving time hurts accuracy.
MODELFILE_TEMPLATE = """# Ollama Modelfile for the fine-tuned 2D floor plan detection VLM
FROM {model_path}

# Deterministic decoding for structured JSON
PARAMETER temperature 0
PARAMETER top_p 1
PARAMETER num_ctx 8192
PARAMETER num_predict 2048
"""


def _abs(p: str) -> Path:
    path = Path(p)
    return path if path.is_absolute() else PROJECT_ROOT / path


def create_modelfile(model_path: str, output_modelfile: str = "Modelfile"):
    out = _abs(output_modelfile)
    out.write_text(MODELFILE_TEMPLATE.format(model_path=Path(model_path).as_posix()), encoding="utf-8")
    print(f"Generated Ollama Modelfile at {out}")
    print("\nTo build and run in Ollama:")
    print(f"  ollama create floorplan-vlm -f \"{out}\"")
    print("  python src/inference.py --engine ollama --model floorplan-vlm --image data/samples/sample_01.png")


def merge_lora_weights(base_model_id: str, adapter_path: str, output_merged_dir: str) -> Path:
    import torch
    from peft import PeftModel
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    adapter = Path(resolve_path(adapter_path))
    if not adapter.is_dir():
        raise SystemExit(f"Adapter not found: {adapter}. Train first (src/train.py) or pass --adapter.")
    out_dir = _abs(output_merged_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading base model {base_model_id} (fp16, CPU) ...")
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(base_model_id, torch_dtype=torch.float16,
                                                              low_cpu_mem_usage=True)
    print(f"Loading adapter {adapter} ...")
    model = PeftModel.from_pretrained(base, str(adapter))
    print("Merging LoRA weights ...")
    merged = model.merge_and_unload()

    print(f"Saving merged model to {out_dir} ...")
    merged.save_pretrained(out_dir, safe_serialization=True)
    # Prefer the processor saved with the adapter (same min/max pixels as training)
    proc_src = str(adapter) if (adapter / "preprocessor_config.json").exists() else base_model_id
    AutoProcessor.from_pretrained(proc_src).save_pretrained(out_dir)
    print("Merge complete!")
    return out_dir


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Export fine-tuned model for Ollama")
    parser.add_argument("--base_model", type=str, default="Qwen/Qwen2.5-VL-3B-Instruct")
    parser.add_argument("--adapter", type=str, default="checkpoints/qwen_floorplan_lora/final_adapter")
    parser.add_argument("--merged_dir", type=str, default="checkpoints/merged_fp16")
    parser.add_argument("--modelfile", type=str, default="Modelfile")
    parser.add_argument("--merge", action="store_true", help="Merge the LoRA adapter into the base model")
    args = parser.parse_args()

    if args.merge:
        merged_dir = merge_lora_weights(args.base_model, args.adapter, args.merged_dir)
        create_modelfile(str(merged_dir), args.modelfile)
    else:
        print("No --merge given: generating a Modelfile for the BASE model from the Ollama library (not fine-tuned).")
        create_modelfile("qwen2.5vl:3b", args.modelfile)

In [ ]:
%%writefile tests/test_pipeline.py
"""
Lightweight tests (no GPU / no model download needed).
Run:  python -m pytest tests -q      or      python tests/test_pipeline.py
"""

import json
import random
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))

from schema import extract_json_from_response, validate_floorplan_json, to_target_text, wall_bbox  # noqa: E402
from evaluate import calculate_iou, match_boxes, evaluate_predictions  # noqa: E402
from dataset_prep import generate_synthetic_floorplan  # noqa: E402
from visualize import build_wall_segments, draw_2d_detections, render_3d_extrusion  # noqa: E402


def _sample():
    tmp = Path(tempfile.mkdtemp())
    return generate_synthetic_floorplan(1, tmp, random.Random(0))


def test_generated_ground_truth_is_valid_and_normalized():
    for seed in range(20):
        item = generate_synthetic_floorplan(seed, Path(tempfile.mkdtemp()), random.Random(seed))
        ok, errs = validate_floorplan_json(item["ground_truth"])
        assert ok, errs


def test_extract_json_variants():
    obj = {"rooms": [], "walls": [], "doors": [], "windows": []}
    s = to_target_text(obj)
    assert extract_json_from_response(s) == obj
    assert extract_json_from_response(f"```json\n{s}\n```") == obj
    assert extract_json_from_response(f"Sure! Here it is:\n{s}\nHope that helps") == obj
    try:
        extract_json_from_response("no json here")
        assert False
    except ValueError:
        pass


def test_validation_catches_bad_boxes():
    bad = {"rooms": [{"id": "r", "bbox": [1, 2, 3]}], "walls": [{}], "doors": [{"bbox": ["a", 1, 2, 3]}],
           "windows": [{"bbox": [0, 0, 2000, 10]}]}
    ok, errs = validate_floorplan_json(bad)
    assert not ok and len(errs) >= 4


def test_iou_and_matching():
    assert calculate_iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0
    assert calculate_iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
    assert abs(calculate_iou([0, 0, 10, 10], [0, 5, 10, 15]) - 1 / 3) < 1e-9
    gts = [{"bbox": [0, 0, 10, 10], "type": "bedroom"}, {"bbox": [20, 20, 30, 30], "type": "kitchen"}]
    preds = [{"bbox": [20, 20, 30, 30], "type": "kitchen"}, {"bbox": [0, 0, 10, 10], "type": "kitchen"},
             {"bbox": [50, 50, 60, 60]}]
    tp, fp, fn, ious, types = match_boxes(preds, gts, "rooms", 0.5)
    assert (tp, fp, fn) == (2, 1, 0)
    assert sorted(types) == [False, True]


def test_walls_without_bbox_are_matched_via_endpoints():
    assert wall_bbox({"start": [100, 50], "end": [100, 500]}) is not None
    gt = [{"start": [100, 50], "end": [100, 500], "bbox": [50, 95, 500, 105]}]
    pred = [{"start": [100, 50], "end": [100, 500]}]
    assert match_boxes(pred, gt, "walls", 0.5)[0] == 1


def test_evaluate_oracle_is_perfect_and_garbage_is_zero():
    gt = _sample()["ground_truth"]
    m = evaluate_predictions({"a": gt}, {"a": gt})
    assert m["macro"]["f1@0.50"] == 1.0 and m["schema_validity_rate_percent"] == 100.0
    m = evaluate_predictions({"a": gt}, {"a": "garbage"})
    assert m["macro"]["f1@0.50"] == 0.0 and m["json_parse_rate_percent"] == 0.0
    m = evaluate_predictions({"a": gt}, {})          # missing prediction
    assert m["macro"]["recall@0.50"] == 0.0


def test_wall_cutting():
    # 10 m horizontal wall with a 1 m door in the middle
    solids = build_wall_segments([(0, 10, 0, 0.2)], [((4.5, 5.5, -0.1, 0.3), "door")], 2.8)
    assert len(solids) == 3                           # left wall, lintel, right wall
    lintel = [s for s in solids if s[4] > 0][0]
    assert abs(lintel[4] - 2.1) < 1e-9


def test_visualization_runs():
    item = _sample()
    out = Path(tempfile.mkdtemp())
    draw_2d_detections(item["abs_image_path"], item["ground_truth"], str(out / "o.png"))
    render_3d_extrusion(item["ground_truth"], str(out / "d.png"))
    assert (out / "o.png").exists() and (out / "d.png").exists()


if __name__ == "__main__":
    fns = [v for k, v in dict(globals()).items() if k.startswith("test_")]
    for fn in fns:
        fn()
        print("PASS", fn.__name__)
    print(f"\nAll {len(fns)} tests passed.")

In [ ]:
!python tests/test_pipeline.py

## 3. Generate the dataset (train / val / test = 96 / 12 / 12)

In [ ]:
!python src/dataset_prep.py --samples 120

In [ ]:
import json, matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i in zip(axes, [1, 2, 3]):
    ax.imshow(Image.open(f"data/samples/sample_{i:02d}.png")); ax.set_title(f"test sample {i}"); ax.axis("off")
plt.show()
print(json.dumps(json.load(open("data/samples/sample_01.json")), indent=1)[:800], "...")

## 4. Verify one training batch (label masking) before training

In [ ]:
import sys; sys.path.insert(0, "src")
from transformers import AutoProcessor
from train import load_dataset_from_jsonl, FloorplanCollator, MIN_PIXELS, MAX_PIXELS

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
ds = load_dataset_from_jsonl("data/processed/train.jsonl")
batch = FloorplanCollator(processor)([ds[0]])
for k, v in batch.items():
    print(k, tuple(v.shape))
lab = batch["labels"][0]
print("total tokens:", lab.numel(), "| trained answer tokens:", int((lab != -100).sum()))
print("decoded target:", processor.tokenizer.decode(lab[lab != -100])[:200], "...")

## 5. Fine-tune with QLoRA
~96 samples × 3 epochs; roughly 20–40 min on a T4.

In [ ]:
!python src/train.py --epochs 3 --batch_size 1 --gradient_accumulation_steps 8

## 6. Evaluate the fine-tuned model on the unseen test split
Compare against the base model to show the effect of fine-tuning (use `--limit` to save time).

In [ ]:
!python src/evaluate.py --run_inference --load_in_4bit --out data/evaluation_results.json

In [ ]:
# Optional baseline: base model WITHOUT the adapter (expect much lower scores)
!python src/evaluate.py --run_inference --load_in_4bit --adapter none --limit 4 --out data/evaluation_baseline.json

## 7. Visualize a prediction (2D overlay + 3D extrusion)

In [ ]:
!python src/inference.py --image data/samples/sample_01.png --load_in_4bit --output_json data/samples/pred_01.json --visualize
from IPython.display import Image as IPImage, display
display(IPImage("data/samples/pred_01_overlay.png", width=550))
display(IPImage("data/samples/pred_01_3d.png", width=650))

## 8. Download adapter + results

In [ ]:
!zip -qr floorplan_results.zip checkpoints/qwen_floorplan_lora/final_adapter data/evaluation_results.json \
    data/evaluation_baseline.json data/predictions_test.jsonl data/samples
from google.colab import files
files.download("floorplan_results.zip")
print("Unzip into your local project folder (keeps the same paths).")